<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/03_ethereum_smart_contracts/notebooks/NB01_Ethereum_Code_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lesson 3: Ethereum and Smart Contracts

This notebook covers Ethereum smart contract interaction:
- Connecting to Ethereum networks (testnet)
- Reading blockchain data
- Interacting with deployed contracts
- Calling contract functions
- Understanding gas and transactions

**IMPORTANT**: This notebook uses a public testnet, so it works on Google Colab!

## Learning Objectives
By the end of this notebook, you will:
- Connect to an Ethereum testnet using Web3.py
- Read blockchain data (blocks, transactions, balances)
- Understand smart contract ABIs and interaction patterns
- Calculate transaction costs using gas mechanics
- Explore account balances and contract events

## Setup

Install web3.py for Ethereum interaction.

In [ ]:
!pip install -q web3

In [ ]:
from web3 import Web3
import json
import time

print("All imports successful!")

## Part 1: Connecting to Ethereum

We'll connect to the Sepolia testnet using a free public RPC endpoint.

**Why Sepolia?**
- Free testnet (no real money needed)
- Public RPC nodes available
- Works perfectly in Google Colab

In [ ]:
# Connect to Sepolia testnet via public RPC
PROVIDER_URL = "https://rpc.sepolia.org"
# Alternative: "https://sepolia.infura.io/v3/YOUR_PROJECT_ID"

w3 = Web3(Web3.HTTPProvider(PROVIDER_URL))

# Verify connection
if w3.is_connected():
    print("✓ Successfully connected to Ethereum!")
    print(f"  Network: Sepolia Testnet")
    print(f"  Chain ID: {w3.eth.chain_id}")
    print(f"  Latest block: {w3.eth.block_number:,}")
else:
    print("✗ Connection failed")

## Part 2: Reading Blockchain Data

Let's explore the Ethereum blockchain by reading some data.

In [ ]:
# Get latest block information
latest_block = w3.eth.get_block('latest')

print("=" * 70)
print("LATEST BLOCK INFORMATION")
print("=" * 70)
print(f"Block Number: {latest_block['number']:,}")
print(f"Timestamp: {time.ctime(latest_block['timestamp'])}")
print(f"Hash: {latest_block['hash'].hex()}")
print(f"Parent Hash: {latest_block['parentHash'].hex()}")
print(f"Transactions: {len(latest_block['transactions'])}")
print(f"Gas Used: {latest_block['gasUsed']:,}")
print(f"Gas Limit: {latest_block['gasLimit']:,}")
print("=" * 70)

### TODO Exercise 1

Explore the blockchain:
1. Get information for a specific block number
2. Look at transaction count over multiple blocks
3. Compare gas usage across blocks

In [ ]:
# TODO: Explore blockchain data
# block_number = w3.eth.block_number - 10  # 10 blocks ago
# old_block = w3.eth.get_block(block_number)
# print(f"Block {block_number} had {len(old_block['transactions'])} transactions")

## Part 3: Examining a Real Transaction

Let's look at a transaction from the latest block.

In [ ]:
# Get the latest block with full transaction details
latest_block_full = w3.eth.get_block('latest', full_transactions=True)

if len(latest_block_full['transactions']) > 0:
    # Get first transaction
    tx = latest_block_full['transactions'][0]
    
    print("=" * 70)
    print("SAMPLE TRANSACTION")
    print("=" * 70)
    print(f"Hash: {tx['hash'].hex()}")
    print(f"From: {tx['from']}")
    print(f"To: {tx['to']}")
    print(f"Value: {w3.from_wei(tx['value'], 'ether')} ETH")
    print(f"Gas: {tx['gas']:,}")
    print(f"Gas Price: {w3.from_wei(tx['gasPrice'], 'gwei')} Gwei")
    print(f"Nonce: {tx['nonce']}")
    print("=" * 70)
else:
    print("No transactions in latest block. Try again in a moment!")

## Part 4: Smart Contract Interaction

Now let's define a simple storage contract and understand how to interact with it.

Here's the Solidity code we're working with:

```solidity
contract SimpleStorage {
    uint256 private storedValue;
    
    function setValue(uint256 newValue) public {
        storedValue = newValue;
    }
    
    function getValue() public view returns (uint256) {
        return storedValue;
    }
}
```

In [ ]:
# SimpleStorage contract ABI (Application Binary Interface)
# This tells web3.py how to interact with the contract
CONTRACT_ABI = json.loads('''[
    {
        "inputs": [],
        "stateMutability": "nonpayable",
        "type": "constructor"
    },
    {
        "inputs": [{"internalType": "uint256", "name": "newValue", "type": "uint256"}],
        "name": "setValue",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "getValue",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function"
    }
]''')

print("Contract ABI loaded!")
print("\nThe ABI defines two functions:")
print("  - setValue(uint256): Store a number (costs gas)")
print("  - getValue(): Read the stored number (free, view-only)")

## Part 5: Understanding Gas

**Gas** is the fee required to execute operations on Ethereum.

Key concepts:
- **Gas**: Unit of computational work
- **Gas Price**: How much you pay per gas unit (in Gwei)
- **Gas Limit**: Maximum gas you're willing to spend
- **Transaction Cost** = Gas Used × Gas Price

Types of operations:
- **View functions** (like `getValue`): FREE - just read data
- **State-changing functions** (like `setValue`): COST GAS - modify blockchain

In [ ]:
# Get current gas price
gas_price_wei = w3.eth.gas_price
gas_price_gwei = w3.from_wei(gas_price_wei, 'gwei')

print("=" * 70)
print("CURRENT GAS INFORMATION")
print("=" * 70)
print(f"Current Gas Price: {gas_price_gwei:.2f} Gwei")
print(f"\nEstimated transaction costs:")
print(f"  Simple transfer (21,000 gas): {w3.from_wei(21000 * gas_price_wei, 'ether'):.6f} ETH")
print(f"  Contract interaction (~50,000 gas): {w3.from_wei(50000 * gas_price_wei, 'ether'):.6f} ETH")
print(f"  Complex contract (~200,000 gas): {w3.from_wei(200000 * gas_price_wei, 'ether'):.6f} ETH")
print("=" * 70)

## Part 6: Contract Interaction Example

Here's how you would interact with a deployed contract. This is demonstration code - you'd need an actual deployed contract address and a funded account to execute transactions.

In [ ]:
# Example contract interaction (read-only)
# NOTE: This is a demonstration. To actually send transactions, you need:
# 1. A deployed contract address
# 2. A private key with testnet ETH

print("=" * 70)
print("CONTRACT INTERACTION EXAMPLE")
print("=" * 70)
print("""
To interact with a contract, you would:

1. CREATE CONTRACT INSTANCE:
   contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=CONTRACT_ABI)

2. CALL VIEW FUNCTION (free, no transaction):
   value = contract.functions.getValue().call()

3. SEND TRANSACTION (costs gas):
   tx_hash = contract.functions.setValue(42).transact({
       'from': YOUR_ADDRESS,
       'gas': 100000,
       'gasPrice': w3.eth.gas_price
   })

4. WAIT FOR CONFIRMATION:
   receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
   
5. CHECK STATUS:
   if receipt['status'] == 1:
       print("Transaction successful!")
""")
print("=" * 70)

## Part 7: Exploring Account Balances

Let's check the balance of a well-known Ethereum address.

In [ ]:
# Check balance of Ethereum Foundation address (example)
# This is a well-known public address
example_address = "0x00000000219ab540356cBB839Cbe05303d7705Fa"

balance_wei = w3.eth.get_balance(example_address)
balance_eth = w3.from_wei(balance_wei, 'ether')

print(f"Address: {example_address}")
print(f"Balance: {balance_eth:.4f} ETH")
print(f"Balance (Wei): {balance_wei:,}")

### TODO Exercise 2

Explore Ethereum accounts:
1. Check the balance of different addresses
2. Look up transaction history for an address using a block explorer
3. Calculate how much a transaction would cost at current gas prices

In [ ]:
# TODO: Check balances of different addresses
# your_address = "0x..."
# balance = w3.eth.get_balance(your_address)
# print(f"Balance: {w3.from_wei(balance, 'ether')} ETH")

## Part 8: Understanding Events

Smart contracts emit **events** to log important occurrences. Events are cheaper than storing data and can be listened to by applications.

In [ ]:
print("=" * 70)
print("SMART CONTRACT EVENTS")
print("=" * 70)
print("""
Events are emitted by contracts to log important actions.

Example event in Solidity:
    event ValueChanged(uint256 indexed oldValue, uint256 indexed newValue);

When setValue() is called:
    emit ValueChanged(storedValue, newValue);

To listen for events in Python:
    event_filter = contract.events.ValueChanged.create_filter(
        fromBlock='latest'
    )
    events = event_filter.get_all_entries()

Events allow:
  - Cheap logging (cheaper than storage)
  - Notification system for applications
  - Historical data retrieval
""")
print("=" * 70)

## Key Takeaways

1. **Web3.py** connects Python to Ethereum networks
2. **Testnets** (like Sepolia) let you experiment without real money
3. **Public RPC nodes** work great for read-only operations and Colab
4. **Gas** is the fuel for Ethereum transactions
5. **View functions** are free to call (read-only)
6. **State-changing functions** require transactions and cost gas
7. **ABIs** define how to interact with smart contracts
8. **Events** provide cheap logging and notifications

Next lesson: We'll create and deploy our own ERC-20 token!